In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Load only the columns needed for this cleaning step
CSV_PATH = "../en.openfoodfacts.org.products 2.csv"

COLS = [
    "states", "states_tags", "states_en",       # three overlapping state columns
    "categories", "categories_tags", "categories_en",  # three overlapping category columns
]

df = pd.read_csv(
    CSV_PATH,
    sep="\t",
    usecols=COLS,      # only read what we need — saves memory on this large file
    low_memory=False,
    on_bad_lines="skip",
)

In [ ]:
# states_en is the most complete and readable version of the states column.
# states and states_tags carry the same information in a less useful format, so we drop them.
df = df.drop(columns=["states", "states_tags"])
df = df.rename(columns={"states_en": "states"})

# drop allergens_en due to it having only 1 entries that is also invalid
df = df.drop(columns=["allergens_en"])

# drop emb_codes_tags
df = df.drop(columns=["emb_codes_tags"])


In [ ]:
# Strings that look like data but are actually missing values in disguise
INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.str.strip()                                       # remove surrounding whitespace
    s = s.replace("", np.nan)                              # empty strings → NaN
    s = s.where(~s.str.lower().isin(INVALID), np.nan)     # known placeholders → NaN
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), np.nan)  # pure noise chars → NaN
    return s

# Apply vectorised cleaning to all three category columns
for c in ["categories", "categories_tags", "categories_en"]:
    df[c] = clean_series(df[c].astype(str).where(df[c].notna(), np.nan))

In [ ]:
# Merge the three category columns into one, preferring the most readable version.
# Priority: categories_en (English, human-readable) → categories_tags (structured) → categories (raw)
df["categories_final"] = (
    df["categories_en"]
      .fillna(df["categories_tags"])
      .fillna(df["categories"])
)

# Drop the original columns now that they are consolidated
df = df.drop(columns=["categories", "categories_tags", "categories_en"])

In [ ]:
# Quick sanity check — print all remaining columns to confirm the result
for col in df.columns:
    print("\n" + "="*40)
    print(col)